In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/raw/diabetes_clean.csv")
print(df.columns.tolist())
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head()

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
60,2,84.0,NaN,NaN,NaN,NaN,0.304,21
618,9,112.0,82.0,24.0,NaN,28.2,1.282,50
346,1,139.0,46.0,19.0,83.0,28.7,0.654,22
294,0,161.0,50.0,NaN,NaN,21.9,0.254,65
231,6,134.0,80.0,37.0,370.0,46.2,0.238,46


In [ ]:
X_train_clean = X_train.fillna(X_train.median())
X_test_clean = X_test.fillna(X_train.median())

In [ ]:

model_base = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='logloss'
)
model_base.fit(X_train_clean, y_train)

y_pred_base = model_base.predict(X_test_clean)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Accuracy modelo base: {acc_base:.4f}")
print(classification_report(y_test, y_pred_base))

In [ ]:



learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
acc_lr = []
for lr in learning_rates:
    m = XGBClassifier(n_estimators=100, learning_rate=lr, max_depth=5, random_state=42, eval_metric='logloss')
    m.fit(X_train_clean, y_train)
    acc_lr.append(accuracy_score(y_test, m.predict(X_test_clean)))


n_estimators_list = [50, 100, 150, 200, 300]
acc_ne = []
for n in n_estimators_list:
    m = XGBClassifier(n_estimators=n, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
    m.fit(X_train_clean, y_train)
    acc_ne.append(accuracy_score(y_test, m.predict(X_test_clean)))


spw_values = [1, 2, 3, 4, 5]
acc_spw = []
for spw in spw_values:
    m = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5,
                      scale_pos_weight=spw, random_state=42, eval_metric='logloss')
    m.fit(X_train_clean, y_train)
    acc_spw.append(accuracy_score(y_test, m.predict(X_test_clean)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(learning_rates, acc_lr, marker='o', color='steelblue', linewidth=2)
axes[0].axhline(acc_base, color='red', linestyle='--', label=f'Base ({acc_base:.3f})')
axes[0].set_title('Impacto de learning_rate')
axes[0].set_xlabel('learning_rate')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_estimators_list, acc_ne, marker='s', color='darkorange', linewidth=2)
axes[1].axhline(acc_base, color='red', linestyle='--', label=f'Base ({acc_base:.3f})')
axes[1].set_title('Impacto de n_estimators')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(spw_values, acc_spw, marker='^', color='seagreen', linewidth=2)
axes[2].axhline(acc_base, color='red', linestyle='--', label=f'Base ({acc_base:.3f})')
axes[2].set_title('Impacto de scale_pos_weight')
axes[2].set_xlabel('scale_pos_weight')
axes[2].set_ylabel('Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Exploración de Hiperparámetros — XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'scale_pos_weight': [1, 2, 3]  
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid,
    cv=5,
    scoring='recall'  
)
grid_search.fit(X_train_clean, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor recall (CV=5): {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_clean)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"\nAccuracy modelo optimizado: {acc_best:.4f}")
print(f"Mejora sobre modelo base: +{(acc_best - acc_base)*100:.2f}%")
print(classification_report(y_test, y_pred_best))


import os
os.makedirs('models', exist_ok=True)
model_filename = "models/boosting_classifier_final.sav"
pickle.dump(best_model, open(model_filename, "wb"))
print(f"Modelo guardado en {model_filename}")

After testing three different models, I have chosen Boosting (XGBoost) as the best solution for predicting diabetes. While the Decision Tree was simple and the Random Forest improved stability, the Boosting model achieved the highest accuracy ($78.5\%$) by sequentially correcting the errors of previous trees. My analysis shows that while the model is very accurate at identifying patients without diabetes (Class 0), it also outperformed the other models in the more difficult task of identifying patients with diabetes (Class 1). Because of this superior precision and its ability to handle complex patterns in the medical data, Boosting is the most reliable choice for this project.